# Underwriter vs Lending Club

`int_rate` is set by Lending Club and reflects its own risk assessment. Including it tests
performance with that assessment available; excluding it tests the borrower data independently.

This notebook compares three feature sets: Lending Club's verdict (`int_rate` and `grade`),
the borrower data available at application, and the union of the two.

In [1]:
import pandas as pd

from lightgbm import LGBMRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score
from sklearn.pipeline import Pipeline

from credit_risk.data import load_loans
from credit_risk.split import out_of_time_split
from credit_risk.model import (
    build_logistic,
    build_lgbm,
    build_tree_preprocessor,
    LC_VERDICT_NUMERIC, LC_VERDICT_CATEGORICAL,
    UNDERWRITER_NUMERIC, UNDERWRITER_CATEGORICAL,
)
from credit_risk.evaluate import discrimination_metrics

TARGET = "target_bad"

df = load_loans()
train, val, test = out_of_time_split(df)

# Check the split against the target proportions of 55%, 20%, and 25%.
n = len(df)
print(f"train {len(train)/n:.1%}  val {len(val)/n:.1%}  test {len(test)/n:.1%}")

train 53.0%  val 21.8%  test 25.2%


In [2]:
# Reuse the feature definitions from model.py.
LC_NUM, LC_CAT = LC_VERDICT_NUMERIC, LC_VERDICT_CATEGORICAL
UW_NUM, UW_CAT = UNDERWRITER_NUMERIC, UNDERWRITER_CATEGORICAL

len(UW_NUM + UW_CAT), len(LC_NUM + LC_CAT)

(27, 2)

## Excluded features

- **Leakage** (`total_pymnt`, `recoveries`, `last_fico_*`): these values were not available at
  origination and are removed in SQL before `stg.loans_clean` is created.
- **LC verdict** (`int_rate`, `grade`, `installment`): these variables reflect Lending Club's
  assessment and are included only in the relevant feature sets.
- **Unavailable** (`il_util`, `open_acc_6m`): these fields were introduced around 2015 and are
  empty during the training period.

SQL removes features that were unavailable at origination. The Python feature lists define
the variables used in each comparison.

## Missing values

Several borrower features contain missing values. The numeric pipeline replaces them with the
median and adds a missing-value indicator. The following comparison checks whether missingness
is associated with a different bad rate for the two features with the most missing values.

In [3]:
# Count missing values in the underwriter features.
df[UW_NUM + UW_CAT].isnull().sum().loc[lambda s: s > 0].sort_values(ascending=False)

mths_since_last_delinq        359458
emp_length                     42891
pub_rec_bankruptcies             697
revol_util                       404
collections_12_mths_ex_med        56
chargeoff_within_12_mths          56
loan_to_income                    52
tax_liens                         39
dti                               32
active_acct_ratio                  1
dtype: int64

In [4]:
# Compare bad rates for missing and present values in the training set.
def missingness(col):
    status = train[col].isnull().map({True: "missing", False: "present"})
    out = train.groupby(status).agg(loans=(TARGET, "size"), bad_rate=(TARGET, "mean"))
    out["share"] = out["loans"] / out["loans"].sum()
    return out


print(missingness("emp_length"))

print()
print(missingness("mths_since_last_delinq"))

             loans  bad_rate     share
emp_length                            
missing      19816   0.19671  0.052813
present     355396   0.12862  0.947187

                         loans  bad_rate     share
mths_since_last_delinq                            
missing                 199677  0.129134  0.532171
present                 175535  0.135722  0.467829


Rows with missing `emp_length` have a higher bad rate than rows where it is present, so the
missing-value indicator preserves useful information. The bad rates for
`mths_since_last_delinq` are similar, suggesting that its missing-value indicator contributes
less information.

## Three feature sets and two models

Fit logistic regression and LightGBM on each feature set and evaluate them on validation data:

1. `LC_*`: Lending Club's verdict, used as the benchmark.
2. `UW_*`: borrower data only.
3. Union: borrower data together with Lending Club's verdict.

PR AUC and Brier score are reported alongside ROC AUC. PR AUC is useful given the 15% event
rate, while the Brier score measures the accuracy of predicted probabilities. Comparing the two
models also shows whether nonlinear feature interactions improve performance.

In [5]:
def fit_and_score(train, val, numeric, categorical, build_model):
    # Fit on training data and calculate validation metrics.
    cols = numeric + categorical
    pipe = build_model(numeric, categorical)
    pipe.fit(train[cols], train[TARGET])

    proba = pipe.predict_proba(val[cols])[:, 1]
    y = val[TARGET].values

    metrics = discrimination_metrics(y, proba)
    return {
        "roc_auc": metrics["roc_auc"],
        "pr_auc": metrics["pr_auc"],
        "brier": metrics["brier"],
    }


results_logistic = {
    "lc_verdict":  fit_and_score(train, val, LC_NUM, LC_CAT, build_logistic),
    "underwriter": fit_and_score(train, val, UW_NUM, UW_CAT, build_logistic),
    "union":       fit_and_score(train, val, UW_NUM + LC_NUM, UW_CAT + LC_CAT, build_logistic),
}

results_lgbm = {
    "lc_verdict":  fit_and_score(train, val, LC_NUM, LC_CAT, build_lgbm),
    "underwriter": fit_and_score(train, val, UW_NUM, UW_CAT, build_lgbm),
    "union":       fit_and_score(train, val, UW_NUM + LC_NUM, UW_CAT + LC_CAT, build_lgbm),
}

print("logistic")
print(pd.DataFrame(results_logistic).T.round(4))

print()
print("lgbm")
print(pd.DataFrame(results_lgbm).T.round(4))

logistic
             roc_auc  pr_auc   brier
lc_verdict    0.6787  0.2517  0.1224
underwriter   0.6633  0.2439  0.1224
union         0.6912  0.2665  0.1207

lgbm
             roc_auc  pr_auc   brier
lc_verdict    0.6754  0.2470  0.1217
underwriter   0.6738  0.2532  0.1216
union         0.6962  0.2750  0.1201


The borrower feature set performs slightly below the Lending Club verdict. The union performs
better than the benchmark. LightGBM performs better on the borrower and union sets, while
logistic regression performs better on the Lending Club verdict set.

## Reconstructing the interest rate

The underwriter model performs slightly below the Lending Club verdict model. To estimate how
much of Lending Club's pricing is explained by the available borrower features, an
`LGBMRegressor` predicts `int_rate` and is evaluated with R^2:

    R^2 = 1 - SS_res / SS_tot     SS_res = sum (y - y_hat)^2,  SS_tot = sum (y - y_bar)^2

R^2 equals 1 for perfect predictions and 0 when the model performs no better than predicting the
mean interest rate.

In [6]:
reg = Pipeline([
    ("prep", build_tree_preprocessor(UW_NUM, UW_CAT)),
    ("model", LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=31, verbose=-1)),
])
reg.fit(train[UW_NUM + UW_CAT], train["int_rate"])

r2 = r2_score(val["int_rate"], reg.predict(val[UW_NUM + UW_CAT]))
print(f"R^2 reconstructing int_rate: {r2:.2f}")

R^2 reconstructing int_rate: 0.38


The borrower features explain about one third of the variation in `int_rate`. The remaining
variation may depend on information absent from this dataset, such as more detailed credit
bureau data. Despite this limitation, the union model performs better than the Lending Club
verdict model.

## Risk separation within a grade

The union model's performance suggests that borrower variables add information beyond Lending
Club's grade. The following analysis tests whether FICO and DTI separate risk within grade C,
which contains enough observations for a within-grade comparison.

In [7]:
# Compare bad rates across FICO and DTI bands within grade C.
grade_c = train[train["grade"] == "C"]

def bad_rate_by_band(col, bins=5):
    band = pd.qcut(grade_c[col], bins, duplicates="drop")
    return grade_c.groupby(band, observed=True).agg(
        loans=(TARGET, "size"),
        bad_rate=(TARGET, "mean"),
    ).round(3)

print(bad_rate_by_band("fico_range_low"))
print()
print(bad_rate_by_band("dti"))

                  loans  bad_rate
fico_range_low                   
(659.999, 665.0]  22762     0.179
(665.0, 675.0]    23321     0.173
(675.0, 685.0]    18554     0.169
(685.0, 695.0]    12937     0.161
(695.0, 840.0]    17852     0.146

                 loans  bad_rate
dti                             
(-0.001, 10.55]  19091     0.136
(10.55, 15.34]   19087     0.152
(15.34, 19.88]   19113     0.164
(19.88, 25.31]   19055     0.185
(25.31, 39.99]   19080     0.198


Bad rates still vary with FICO and DTI within grade C. Higher FICO values and lower DTI values
are associated with lower bad rates, indicating that these variables retain information not
fully represented by the grade.

## Feature importance

Permutation importance measures how much validation ROC AUC drops when a single feature is
shuffled. It is more reliable than the model's built-in importance when features are correlated,
since that metric can divide credit arbitrarily between related variables. Computing it for both
models on the borrower-only set shows which features each relies on, and whether the engineered
ratios and the derogatory-event counts contribute.

In [8]:
# Permutation importance for both models on the borrower-only feature set.
def permutation_ranking(build_model):
    cols = UW_NUM + UW_CAT
    model = build_model(UW_NUM, UW_CAT).fit(train[cols], train[TARGET])
    result = permutation_importance(
        model, val[cols], val[TARGET],
        scoring="roc_auc", n_repeats=5, random_state=0, n_jobs=-1,
    )
    return pd.Series(result.importances_mean, index=cols)


importance = pd.DataFrame({
    "logistic": permutation_ranking(build_logistic),
    "lgbm": permutation_ranking(build_lgbm),
})
importance.sort_values("lgbm", ascending=False).head(10).round(4)

,logistic,lgbm
fico_range_low,0.0548,0.0578
revol_bal,0.0038,0.0171
dti,0.0141,0.0154
inq_last_6mths,0.0150,0.0130
loan_to_income,0.0226,0.0091
purpose,0.0114,0.0090
annual_inc,0.0001,0.0066
home_ownership,0.0058,0.0063
loan_amnt,0.0024,0.0055
credit_history_months,0.0007,0.0045


In [9]:
# Group features into classes to see what each model relies on.
FEATURE_CLASSES = {
    "fico": ["fico_range_low"],
    "engineered": ["loan_to_income", "active_acct_ratio"],
    "derogatory": [
        "collections_12_mths_ex_med", "tax_liens", "delinq_amnt", "acc_now_delinq",
        "chargeoff_within_12_mths", "pub_rec", "pub_rec_bankruptcies", "delinq_2yrs",
        "mths_since_last_delinq",
    ],
    "debt_profile": [
        "revol_bal", "revol_util", "dti", "total_acc", "open_acc", "loan_amnt",
        "annual_inc", "credit_history_months",
    ],
    "inquiries": ["inq_last_6mths"],
    "categorical": UW_CAT,
}

pd.DataFrame(
    {name: importance.loc[feats].sum() for name, feats in FEATURE_CLASSES.items()}
).T.round(4)

,logistic,lgbm
fico,0.0548,0.0578
engineered,0.0235,0.0131
derogatory,0.0013,0.0023
debt_profile,0.0232,0.0551
inquiries,0.0150,0.0130
categorical,0.0291,0.0245


FICO is the strongest feature for both models. The differences appear in the rest.

The engineered ratios matter more to logistic regression than to LightGBM. `loan_to_income` is a
hand-built interaction, and the linear model cannot form interactions on its own, whereas the
tree recovers the same signal from the raw features. LightGBM instead draws far more from the raw
debt-profile variables, through the nonlinear splits the linear model cannot make.

The derogatory-event counts contribute almost nothing to either model, consistent with their
weak marginal signal seen earlier. They are real but rare, and add little once FICO and the debt
profile are present.

## Conclusions

Borrower data alone performs close to Lending Club's verdict, and the union of both feature sets
performs better than either set alone. The borrower variables explain only part of the variation
in `int_rate`, suggesting that additional pricing inputs are absent from this dataset.

LightGBM performs better than logistic regression on the borrower and union feature sets, but not on the Lending Club verdict set. This is expected because the Lending Club verdict is effectively a single smooth, monotonic feature. Within grade C, FICO and DTI still differentiate bad rates. Within grade C, FICO and DTI continue to separate bad rates.
The selected model's calibration is evaluated in `scripts/train_baseline.py`.